# genQC Paper Evaluation

Reproduces the SRV evaluation figure from the genQC paper and compares multiple trained models on the same metric.

**Paper metric**: for each qubit count (3–8), evaluate on the corresponding dataset and compute accuracy per number-of-entangled-qubits bucket. Visualise as one line per qubit count (Oranges colormap).

## 1. Setup

In [ ]:
import os, sys, random
from pathlib import Path

import hydra
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra.core.global_hydra import GlobalHydra
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from quantum_diffusion.evaluation.evaluator import SRVEvaluator

In [ ]:
# ── Edit only this cell ────────────────────────────────────────────────────────

DATASET_BASE = "./datasets/paper_quditkit"   # datasets live at {DATASET_BASE}/srv_{n}q_dataset
QUBIT_COUNTS  = [3, 4, 5, 6, 7, 8]
SEED          = 1234

# Paper (Extended Data Table III, Fig. 2a): 8192 samples per entanglement bucket.
# Entanglement buckets for n qubits: {0, 2, 3, ..., n}  (no bucket=1 is possible).
# With a balanced dataset, random sampling of SAMPLES_PER_BUCKET * n_buckets is a
# close approximation, but set USE_STRATIFIED=True for exact paper reproduction.
SAMPLES_PER_BUCKET = 8192
USE_STRATIFIED     = True   # False → random sampling of SAMPLES_PER_BUCKET * n_buckets

MODEL_SPECS = [
    {
        "label":     "paper_srv_model",
        "model_dir": "./models/trained/paper_srv_model",
        "hf_repo":   None,
    },
    # Add more models here, e.g.:
    # {"label": "my_finetuned", "model_dir": "./models/trained/my_finetuned", "hf_repo": None},
]

## 2. Evaluate

In [ ]:
import ast
from collections import defaultdict
from my_genQC.inference.sampling import generate_tensors as _generate_tensors


def _build_cfg(dataset_path, model_dir, hf_repo, num_samples):
    GlobalHydra.instance().clear()
    with hydra.initialize(version_base=None, config_path="../conf"):
        cfg = hydra.compose(config_name="config.yaml", overrides=["evaluation=paper_srv"])
    cfg = cfg["evaluation"]
    cfg.dataset   = str(Path(dataset_path).expanduser().resolve())
    cfg.model_dir = str(Path(model_dir).expanduser().resolve()) if model_dir else None
    cfg.hf_repo   = hf_repo
    cfg.num_samples   = int(num_samples)
    cfg.max_gates     = 16    # paper Fig. 2a: max_gates = 16
    cfg.save_output   = False
    cfg.wandb.enable  = False
    return cfg


def _stratified_indices(dataset, samples_per_bucket, seed):
    """Return indices sampled uniformly from each entanglement bucket."""
    rng = random.Random(seed)
    bucket_indices = defaultdict(list)
    for i, label in enumerate(dataset.y):
        text = str(label)
        srv = ast.literal_eval(text[text.find("["):text.find("]") + 1])
        bucket_indices[sum(1 for v in srv if v == 2)].append(i)
    idx = []
    for bucket in sorted(bucket_indices):
        pool = bucket_indices[bucket]
        idx.extend(rng.sample(pool, min(samples_per_bucket, len(pool))))
    return idx


def evaluate_model_all_qubits(model_spec, qubit_counts, dataset_base,
                               samples_per_bucket, use_stratified, seed):
    """Returns dict: num_qubits -> {srv_exact_match_rate, acc_per_entanglement, conversion_rate}."""
    out = {}
    for q in qubit_counts:
        dataset_path = f"{dataset_base}/srv_{q}q_dataset"
        n_buckets    = q - 1  # buckets: 0, 2, 3, ..., q  (skipping 1)
        num_samples  = samples_per_bucket * n_buckets

        cfg = _build_cfg(dataset_path, model_spec.get("model_dir"),
                         model_spec.get("hf_repo"), num_samples)

        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        evaluator = SRVEvaluator(config=cfg)

        if use_stratified:
            # Override generate_tensors to use stratified indices.
            strat_idx = _stratified_indices(evaluator.dataset, samples_per_bucket, seed)
            evaluator.samples = len(strat_idx)
            evaluator.idx     = strat_idx
            prompts = [str(evaluator.dataset.y[i]) for i in strat_idx]
            tensors_out = _generate_tensors(
                pipeline=evaluator.pipeline,
                prompt=prompts,
                samples=evaluator.samples,
                system_size=evaluator.system_size,
                num_of_qubits=evaluator.num_qubits,
                max_gates=evaluator.max_gates,
                g=cfg.model_params.guidance_scale,
                auto_batch_size=cfg.model_params.auto_batch_size,
                enable_params=False,
                no_bar=False,
            )
            decoded  = evaluator.decode_tensors(tensors_out)
            _, t_srv, p_srv = evaluator.validate_and_calculate_srvs(decoded, save_output=False)
        else:
            tensors_out = evaluator.generate_tensors(save_output=False)
            decoded     = evaluator.decode_tensors(tensors_out)
            _, t_srv, p_srv = evaluator.validate_and_calculate_srvs(decoded, save_output=False)

        srv_exact_match_rate, acc_per_entanglement = evaluator.calculate_metrics(t_srv, p_srv)
        conversion_rate = len(t_srv) / evaluator.samples if evaluator.samples else 0.0

        acc = {k: v for k, v in acc_per_entanglement.items() if v > 0 or k == 0}
        out[q] = {
            "srv_exact_match_rate": srv_exact_match_rate,
            "acc_per_entanglement": acc,
            "conversion_rate": conversion_rate,
        }
        print(f"  {q}q  exact_match={srv_exact_match_rate:.4f}  conversion={conversion_rate:.4f}")
    return out


# results[model_label][num_qubits] = {...}
results = {}

for i, spec in enumerate(MODEL_SPECS):
    print(f"\n=== {spec['label']} ===")
    results[spec["label"]] = evaluate_model_all_qubits(
        model_spec=spec,
        qubit_counts=QUBIT_COUNTS,
        dataset_base=DATASET_BASE,
        samples_per_bucket=SAMPLES_PER_BUCKET,
        use_stratified=USE_STRATIFIED,
        seed=SEED + i,
    )

print("\nDone.")

## 3. Paper Figure

One plot per model. Each line = one qubit count, colour = qubit count (discrete Oranges colormap). This reproduces the figure from the genQC paper.

In [ ]:
def paper_figure(model_label, qubit_results, ax=None):
    qubits = np.array(sorted(qubit_results.keys()))
    cmap   = plt.get_cmap("Oranges", len(qubits))
    bounds = np.arange(qubits.min() - 0.5, qubits.max() + 1.5)
    norm   = mpl.colors.BoundaryNorm(bounds, cmap.N)

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3.6), dpi=150)
    else:
        fig = ax.get_figure()

    for q in qubits:
        acc = qubit_results[q]["acc_per_entanglement"]
        xs  = sorted(acc.keys())
        ys  = [acc[x] for x in xs]
        ax.plot(xs, ys, marker="o", markersize=3, color=cmap(norm(q)), alpha=0.95)

    sm   = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02, ticks=qubits, boundaries=bounds)
    cbar.set_label("Qubits")

    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Entangled qubits")
    ax.set_ylabel("Accuracy")
    ax.set_title(model_label)
    ax.grid(True, alpha=0.35)
    return fig, ax


n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 3.6), dpi=150, squeeze=False)

for ax, (label, qubit_results) in zip(axes[0], results.items()):
    paper_figure(label, qubit_results, ax=ax)

plt.tight_layout()
plt.show()

## 4. Model Comparison

Summary table (exact-match rate per model × qubit count) and a comparison line plot (mean accuracy across entanglement buckets per qubit count).

In [ ]:
# ── Summary table: rows = models, columns = qubit counts ──────────────────────
# Shows exact-match rate and (conversion rate) per qubit count.
rows = []
for label, qubit_results in results.items():
    row = {"model": label}
    for q in QUBIT_COUNTS:
        if q in qubit_results:
            r = qubit_results[q]
            row[f"{q}q acc"] = round(r["srv_exact_match_rate"], 4)
            row[f"{q}q conv"] = round(r["conversion_rate"], 4)
        else:
            row[f"{q}q acc"] = float("nan")
            row[f"{q}q conv"] = float("nan")
    rows.append(row)

df = pd.DataFrame(rows).set_index("model")
# Paper (Fig 2a context): conversion rate ≈ 99.6%
display(df)

In [ ]:
# ── Comparison plot: mean accuracy per qubit count, one line per model ─────────
if len(results) > 1:
    fig, ax = plt.subplots(figsize=(6, 3.8), dpi=150)
    for label, qubit_results in results.items():
        qs   = sorted(qubit_results.keys())
        means = [float(np.mean(list(qubit_results[q]["acc_per_entanglement"].values()))) for q in qs]
        ax.plot(qs, means, marker="o", markersize=4, linewidth=1.8, label=label)

    ax.set_xlabel("Qubit count")
    ax.set_ylabel("Mean accuracy (over entanglement buckets)")
    ax.set_xticks(QUBIT_COUNTS)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.35)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Only one model — nothing to compare.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=140)
plotted = False

for label, result in results_by_model.items():
    bucket_metrics = result["micro_accuracy_by_entangled_qubits"]
    if not bucket_metrics:
        continue
    xs = sorted(bucket_metrics)
    ys = [bucket_metrics[x] for x in xs]
    ax.plot(xs, ys, marker="s", linestyle="--", linewidth=1.4, markersize=4, label=label)
    plotted = True

ax.set_xlabel("Entangled qubits")
ax.set_ylabel("Micro exact-match accuracy")
ax.set_title("Current evaluator bucket accuracy")
ax.set_ylim(0, 1.05)
if plotted:
    ax.legend(title="Model")
else:
    ax.text(0.5, 0.5, "No valid decodes available.", ha="center", va="center", transform=ax.transAxes)
plt.tight_layout()
plt.show()
